# SOTA-Style Temporal and Graph-Enhanced Neural Models

This notebook evaluates compact but strong temporal neural baselines: LSTM, GRU, TCN, Transformer encoder, graph-enhanced temporal convolution, Graph WaveNet-inspired dilation, PatchTST-style patching, and a compact attention model inspired by Temporal Fusion Transformers. The models use fixed-length windows, early stopping, deterministic seeds, and PyTorch MPS when available.

In [1]:
import os, json, math, time, zipfile, subprocess, warnings, textwrap, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 17,
    "axes.titlesize": 18,
    "axes.labelsize": 17,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "figure.titlesize": 18,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 350,
    "figure.dpi": 350,
})

PALETTE = {
    "blue": "#2F6B8F",
    "orange": "#C77C2B",
    "green": "#4F7F3A",
    "purple": "#6D5A8D",
    "red": "#B45A55",
    "gray": "#6E7378",
    "light_gray": "#E7EAED",
    "teal": "#4A8C8A",
    "gold": "#C9A227",
    "black": "#222222",
    "white": "#FFFFFF",
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "Data" / "Raw"
DATA_UNIFIED = PROJECT_ROOT / "Data" / "Unified"
DATA_EXTERNAL = PROJECT_ROOT / "Data" / "External_Test"
TABLES = PROJECT_ROOT / "Tables"
FIGURES = PROJECT_ROOT / "Figures"
MODELS = PROJECT_ROOT / "Models"
REPORTS = PROJECT_ROOT / "Reports"
for p in [DATA_RAW, DATA_UNIFIED, DATA_EXTERNAL, TABLES, FIGURES, MODELS, REPORTS, PROJECT_ROOT / "Drawio_Preview"]:
    p.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed: {RANDOM_SEED}")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, balanced_accuracy_score, roc_auc_score, average_precision_score, brier_score_loss

Project root: /Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1
Random seed: 42


## Window Dataset Construction

The neural comparison uses a computationally feasible selected-line subset and 24-hour lookback windows. Targets remain the 1-hour future line flow and 1-hour Critical-risk label.

In [2]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)

model_df = pd.read_parquet(DATA_UNIFIED / "unified_powergrid_modeling_dataset.parquet")
selected_lines = pd.read_csv(DATA_UNIFIED / "selected_lines_metadata.csv")
line_subset = selected_lines.head(24)["line_id"].astype(str).tolist()
seq_features = [
    "line_flow", "absolute_line_flow", "total_load", "total_generation", "load_ramp", "generation_ramp",
    "neighborhood_mean_flow", "graph_smoothed_flow", "risk_proximity_score", "line_centrality", "stress_regime"
]
seq_features = [c for c in seq_features if c in model_df.columns]
LOOKBACK = 24
MAX_PER_SPLIT = {"train": 26000, "validation": 7000, "test": 9000, "external_test": 9000}

work = model_df[model_df["line_id"].astype(str).isin(line_subset)].sort_values(["scenario_group", "line_id", "time_index"]).copy()
scaler_x = StandardScaler().fit(work[work["split"] == "train"][seq_features].replace([np.inf, -np.inf], np.nan).fillna(0))
y_scaler = StandardScaler().fit(work[work["split"] == "train"][["future_line_flow_h1"]])

def build_windows(split):
    Xs, yr, yc, rows = [], [], [], []
    sub_all = work[work["split"] == split]
    for (_, _), grp in sub_all.groupby(["scenario_group", "line_id"], sort=False):
        grp = grp.sort_values("time_index")
        Xmat = scaler_x.transform(grp[seq_features].replace([np.inf, -np.inf], np.nan).fillna(0))
        yreg = y_scaler.transform(grp[["future_line_flow_h1"]]).ravel()
        ycls = grp["risk_binary_h1"].to_numpy(dtype=np.float32)
        row_ids = grp["row_id"].to_numpy()
        for i in range(LOOKBACK, len(grp)):
            Xs.append(Xmat[i-LOOKBACK:i])
            yr.append(yreg[i])
            yc.append(ycls[i])
            rows.append(row_ids[i])
    Xs = np.asarray(Xs, dtype=np.float32); yr = np.asarray(yr, dtype=np.float32); yc = np.asarray(yc, dtype=np.float32); rows = np.asarray(rows)
    max_n = MAX_PER_SPLIT.get(split, len(rows))
    if len(rows) > max_n:
        rng = np.random.default_rng(RANDOM_SEED)
        idx = rng.choice(len(rows), max_n, replace=False)
        Xs, yr, yc, rows = Xs[idx], yr[idx], yc[idx], rows[idx]
    return Xs, yr, yc, rows

data = {split: build_windows(split) for split in ["train", "validation", "test", "external_test"]}
for split, (X, yr, yc, rows) in data.items():
    print(split, X.shape, "critical rate", float(yc.mean()))

PyTorch version: 2.11.0
Device: mps


train (26000, 24, 11) critical rate 0.0494999997317791
validation (7000, 24, 11) critical rate 0.041428569704294205
test (9000, 24, 11) critical rate 0.04555555433034897
external_test (9000, 24, 11) critical rate 0.04911111295223236


## Neural Architectures

All architectures produce a regression output and a risk logit. Graph-enhanced variants consume the same graph-neighborhood and topology-aware features constructed in Notebook 02.

In [3]:
class MultiTaskHead(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.reg = nn.Linear(hidden, 1)
        self.cls = nn.Linear(hidden, 1)
    def forward(self, h):
        return self.reg(h).squeeze(-1), self.cls(h).squeeze(-1)

class LSTMModel(nn.Module):
    def __init__(self, d):
        super().__init__(); self.rnn = nn.LSTM(d, 48, batch_first=True); self.head = MultiTaskHead(48)
    def forward(self, x):
        out, _ = self.rnn(x); return self.head(out[:, -1])

class GRUModel(nn.Module):
    def __init__(self, d):
        super().__init__(); self.rnn = nn.GRU(d, 48, batch_first=True); self.head = MultiTaskHead(48)
    def forward(self, x):
        out, _ = self.rnn(x); return self.head(out[:, -1])

class TCNModel(nn.Module):
    def __init__(self, d):
        super().__init__(); self.net = nn.Sequential(nn.Conv1d(d, 48, 3, padding=1), nn.ReLU(), nn.Conv1d(48, 48, 3, padding=2, dilation=2), nn.ReLU()); self.head = MultiTaskHead(48)
    def forward(self, x):
        h = self.net(x.transpose(1,2)).mean(-1); return self.head(h)

class TransformerModel(nn.Module):
    def __init__(self, d):
        super().__init__(); self.proj = nn.Linear(d, 48); enc = nn.TransformerEncoderLayer(d_model=48, nhead=4, dim_feedforward=96, batch_first=True, dropout=0.05); self.enc = nn.TransformerEncoder(enc, num_layers=2); self.head = MultiTaskHead(48)
    def forward(self, x):
        h = self.enc(self.proj(x)).mean(1); return self.head(h)

class STGCNInspired(nn.Module):
    def __init__(self, d):
        super().__init__(); self.mix = nn.Linear(d, 48); self.conv = nn.Conv1d(48, 48, 5, padding=2); self.head = MultiTaskHead(48)
    def forward(self, x):
        z = torch.relu(self.mix(x)).transpose(1,2); h = torch.relu(self.conv(z)).mean(-1); return self.head(h)

class GraphWaveNetInspired(nn.Module):
    def __init__(self, d):
        super().__init__(); self.inp = nn.Conv1d(d, 48, 1); self.c1 = nn.Conv1d(48, 48, 3, padding=2, dilation=2); self.c2 = nn.Conv1d(48, 48, 3, padding=4, dilation=4); self.head = MultiTaskHead(48)
    def forward(self, x):
        z = torch.relu(self.inp(x.transpose(1,2))); z = torch.relu(self.c1(z)); z = torch.relu(self.c2(z)); return self.head(z.mean(-1))

class PatchTSTSimple(nn.Module):
    def __init__(self, d, patch=4):
        super().__init__(); self.patch = patch; self.proj = nn.Linear(d, 48); enc = nn.TransformerEncoderLayer(48, 4, 96, batch_first=True, dropout=0.05); self.enc = nn.TransformerEncoder(enc, 1); self.head = MultiTaskHead(48)
    def forward(self, x):
        b,t,d = x.shape; p = self.patch; x = x[:, :t//p*p, :].reshape(b, t//p, p, d).mean(2); h = self.enc(self.proj(x)).mean(1); return self.head(h)

class TFTCompact(nn.Module):
    def __init__(self, d):
        super().__init__(); self.rnn = nn.LSTM(d, 48, batch_first=True); self.attn = nn.MultiheadAttention(48, 4, batch_first=True); self.gate = nn.Sequential(nn.Linear(48,48), nn.Sigmoid()); self.head = MultiTaskHead(48)
    def forward(self, x):
        z,_ = self.rnn(x); a,_ = self.attn(z,z,z); h = (self.gate(a)*a).mean(1); return self.head(h)

model_builders = {
    "LSTM": LSTMModel,
    "GRU": GRUModel,
    "TCN": TCNModel,
    "Transformer Encoder": TransformerModel,
    "STGCN-inspired compact": STGCNInspired,
    "Graph WaveNet-inspired compact": GraphWaveNetInspired,
    "PatchTST-style compact": PatchTSTSimple,
    "TFT-inspired compact attention": TFTCompact,
}
print("Defined models:", list(model_builders))

Defined models: ['LSTM', 'GRU', 'TCN', 'Transformer Encoder', 'STGCN-inspired compact', 'Graph WaveNet-inspired compact', 'PatchTST-style compact', 'TFT-inspired compact attention']


## Train and Evaluate SOTA-Style Models

Early stopping uses validation loss only. Test and external-test rows are evaluated after training decisions are fixed.

In [4]:
def make_loader(split, batch=256, shuffle=False):
    X, yr, yc, _ = data[split]
    ds = TensorDataset(torch.tensor(X), torch.tensor(yr), torch.tensor(yc))
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

def train_model(name, builder, epochs=5, patience=2):
    model = builder(len(seq_features)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    mse = nn.MSELoss(); bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([4.0], device=device))
    best_state, best_val, bad = None, float("inf"), 0
    history = []
    t0 = time.time()
    for epoch in range(1, epochs+1):
        model.train(); losses=[]
        for xb, yrb, ycb in make_loader("train", shuffle=True):
            xb, yrb, ycb = xb.to(device), yrb.to(device), ycb.to(device)
            opt.zero_grad(); pr, logit = model(xb); loss = mse(pr, yrb) + 0.25*bce(logit, ycb); loss.backward(); opt.step(); losses.append(float(loss.detach().cpu()))
        model.eval(); vloss=[]
        with torch.no_grad():
            for xb, yrb, ycb in make_loader("validation"):
                xb, yrb, ycb = xb.to(device), yrb.to(device), ycb.to(device)
                pr, logit = model(xb); loss = mse(pr, yrb) + 0.25*bce(logit, ycb); vloss.append(float(loss.detach().cpu()))
        tr_loss, va_loss = float(np.mean(losses)), float(np.mean(vloss))
        history.append({"model": name, "epoch": epoch, "train_loss": tr_loss, "validation_loss": va_loss})
        if va_loss < best_val:
            best_val, bad = va_loss, 0
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), time.time()-t0

def evaluate_model(model, split):
    X, yr_scaled, yc, rows = data[split]
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(yr_scaled), torch.tensor(yc)), batch_size=512)
    pred_scaled, logits = [], []
    t0 = time.time()
    model.eval()
    with torch.no_grad():
        for xb, _, _ in loader:
            pr, lo = model(xb.to(device)); pred_scaled.append(pr.cpu().numpy()); logits.append(lo.cpu().numpy())
    infer_time = time.time()-t0
    pred_scaled = np.concatenate(pred_scaled); logits = np.concatenate(logits)
    y_pred = y_scaler.inverse_transform(pred_scaled.reshape(-1,1)).ravel()
    y_true = y_scaler.inverse_transform(yr_scaled.reshape(-1,1)).ravel()
    prob = 1/(1+np.exp(-logits))
    cls_pred = (prob >= 0.5).astype(int)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    out = {
        "split": split,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "normalized_RMSE": rmse/(np.std(y_true)+1e-9),
        "Balanced Accuracy": balanced_accuracy_score(yc, cls_pred),
        "Macro-F1": f1_score(yc, cls_pred, average="macro", zero_division=0),
        "ROC-AUC": roc_auc_score(yc, prob) if len(np.unique(yc)) > 1 else np.nan,
        "PR-AUC": average_precision_score(yc, prob) if len(np.unique(yc)) > 1 else np.nan,
        "Brier Score": brier_score_loss(yc, np.clip(prob,0,1)),
        "inference_time_seconds": infer_time,
    }
    pred_df = pd.DataFrame({"row_id": rows, "split": split, "y_true_flow": y_true, "y_pred_flow": y_pred, "y_true_risk": yc, "y_prob": prob, "y_pred_risk": cls_pred})
    return out, pred_df

all_histories, class_rows, fore_rows, pred_rows, train_summaries = [], [], [], [], []
for name, builder in model_builders.items():
    model, hist, fit_time = train_model(name, builder)
    all_histories.append(hist)
    for split in ["validation", "test", "external_test"]:
        metrics, pdf = evaluate_model(model, split)
        class_rows.append({"model": name, "split": split, "Balanced Accuracy": metrics["Balanced Accuracy"], "Macro-F1": metrics["Macro-F1"], "ROC-AUC": metrics["ROC-AUC"], "PR-AUC": metrics["PR-AUC"], "Brier Score": metrics["Brier Score"], "inference_time_seconds": metrics["inference_time_seconds"]})
        fore_rows.append({"model": name, "split": split, "MAE": metrics["MAE"], "RMSE": metrics["RMSE"], "normalized_RMSE": metrics["normalized_RMSE"], "inference_time_seconds": metrics["inference_time_seconds"]})
        pdf["model"] = name
        pred_rows.append(pdf)
    train_summaries.append({"model": name, "training_time_seconds": fit_time, "epochs_completed": int(hist["epoch"].max()), "best_validation_loss": float(hist["validation_loss"].min()), "device": str(device), "lookback_hours": LOOKBACK})
    print("Finished", name, "training seconds", round(fit_time, 2))

sota_class = pd.DataFrame(class_rows)
sota_fore = pd.DataFrame(fore_rows)
sota_train = pd.DataFrame(train_summaries)
sota_preds = pd.concat(pred_rows, ignore_index=True)
histories = pd.concat(all_histories, ignore_index=True)

sota_class.to_csv(TABLES / "sota_classification_metrics.csv", index=False)
sota_fore.to_csv(TABLES / "sota_forecasting_metrics.csv", index=False)
sota_train.to_csv(TABLES / "sota_model_training_summary.csv", index=False)
sota_preds[sota_preds["split"] == "validation"].to_csv(TABLES / "sota_validation_predictions.csv", index=False)
sota_preds[sota_preds["split"] == "test"].to_csv(TABLES / "sota_test_predictions.csv", index=False)
sota_preds[sota_preds["split"] == "external_test"].to_csv(TABLES / "sota_external_test_predictions.csv", index=False)
histories.to_csv(TABLES / "sota_training_curves_long.csv", index=False)

display(sota_class.sort_values(["split", "Macro-F1"], ascending=[True, False]))
display(sota_fore.sort_values(["split", "MAE"]))
display(sota_train)
print("Saved SOTA metrics and predictions.")

Finished LSTM training seconds 7.71


Finished GRU training seconds 14.46


Finished TCN training seconds 5.74


Finished Transformer Encoder training seconds 14.76


Finished STGCN-inspired compact training seconds 6.99


...

Finished TFT-inspired compact attention training seconds 10.75


,model,split,Balanced Accuracy,Macro-F1,ROC-AUC,PR-AUC,Brier Score,inference_time_seconds
5,GRU,external_test,0.891491,0.803085,0.975042,0.692267,0.035242,0.255124
2,LSTM,external_test,0.886462,0.792898,0.973821,0.683567,0.034684,0.159787
23,TFT-inspired compact attention,external_test,0.876440,0.778432,0.968428,0.627253,0.039340,0.232446
11,Transformer Encoder,external_test,0.771348,0.729698,0.947304,0.499250,0.041843,0.244425
20,PatchTST-style compact,external_test,0.784423,0.719685,0.934764,0.441822,0.050228,1.491754
17,Graph WaveNet-inspired compact,external_test,0.661169,0.700751,0.943024,0.505666,0.033671,0.175796
14,STGCN-inspired compact,external_test,0.680431,0.686002,0.904819,0.375856,0.046341,0.133848
8,TCN,external_test,0.693310,0.682106,0.901812,0.382689,0.050805,0.121584
4,GRU,test,0.891476,0.799402,0.973617,0.756138,0.034241,0.273240
10,Transformer Encoder,test,0.804329,0.788546,0.954858,0.646638,0.030293,0.277323


,model,split,MAE,RMSE,normalized_RMSE,inference_time_seconds
5,GRU,external_test,1.292688,1.843251,0.172556,0.255124
2,LSTM,external_test,1.343197,1.922125,0.179940,0.159787
23,TFT-inspired compact attention,external_test,1.781177,2.609711,0.244309,0.232446
17,Graph WaveNet-inspired compact,external_test,1.829350,2.542991,0.238063,0.175796
14,STGCN-inspired compact,external_test,2.771672,3.931366,0.368036,0.133848
8,TCN,external_test,2.800350,3.948355,0.369626,0.121584
11,Transformer Encoder,external_test,2.843390,4.007993,0.375209,0.244425
20,PatchTST-style compact,external_test,2.960251,4.149302,0.388438,1.491754
4,GRU,test,1.398934,2.002912,0.185279,0.273240
1,LSTM,test,1.459194,2.094480,0.193750,0.171427


,model,training_time_seconds,epochs_completed,best_validation_loss,device,lookback_hours
0,LSTM,7.713047,5,0.064467,mps,24
1,GRU,14.463336,5,0.064970,mps,24
2,TCN,5.743279,5,0.200137,mps,24
3,Transformer Encoder,14.756107,5,0.184697,mps,24
4,STGCN-inspired compact,6.985698,5,0.192688,mps,24
5,Graph WaveNet-inspired compact,7.928707,5,0.119722,mps,24
6,PatchTST-style compact,10.178856,5,0.199231,mps,24
7,TFT-inspired compact attention,10.748521,5,0.098812,mps,24


Saved SOTA metrics and predictions.


## SOTA Figures

The figures compare model performance, learning curves, actual-vs-predicted forecasts, and external-test stability.

In [5]:

# Compact artifact index for SOTA-style model computations. Final manuscript figures are generated in Notebook 06.
artifact_rows = []
for path in [
    TABLES / "sota_classification_metrics.csv",
    TABLES / "sota_forecasting_metrics.csv",
    TABLES / "sota_model_training_summary.csv",
    TABLES / "sota_training_curves_long.csv",
    TABLES / "sota_validation_predictions.csv",
    TABLES / "sota_test_predictions.csv",
    TABLES / "sota_external_test_predictions.csv",
]:
    if path.exists():
        artifact_rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_mb": path.stat().st_size / (1024 ** 2)})
artifact_index = pd.DataFrame(artifact_rows)
display(artifact_index)
print("SOTA-style models trained and saved. Final publication-ready visualizations are centralized in Notebook 06.")


,artifact,size_mb
0,Tables/sota_classification_metrics.csv,0.003364
1,Tables/sota_forecasting_metrics.csv,0.002236
2,Tables/sota_model_training_summary.csv,0.000578
3,Tables/sota_training_curves_long.csv,0.002282
4,Tables/sota_validation_predictions.csv,4.474876
5,Tables/sota_test_predictions.csv,5.355215
6,Tables/sota_external_test_predictions.csv,5.958649


SOTA-style models trained and saved. Final publication-ready visualizations are centralized in Notebook 06.
